In [2]:
import requests
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, GenerationConfig, pipeline
from pyngrok import ngrok
import nest_asyncio
import uvicorn
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import numpy as np
from vllm import LLM, SamplingParams
from transformers import Pipeline, PreTrainedTokenizer



In [3]:
llm = LLM(
    model="cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0",
    tensor_parallel_size=8,
    max_model_len=4096,
    gpu_memory_utilization=0.5,
)
tokenizer = AutoTokenizer.from_pretrained("cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0", trust_remote_cote=True)


/home/ubuntu/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2024-06-10 10:23:07,746	INFO worker.py:1749 -- Started a local Ray instance.


INFO 06-10 10:23:08 llm_engine.py:161] Initializing an LLM engine (v0.4.3) with config: model='cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0', speculative_config=None, tokenizer='cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=8, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), seed=0, served_model_name=cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0)
INFO 06-10 10:23:35 utils.py:618] Found nccl from library libnccl.so.2
INFO 06-10 10:23:35 pynccl.py:65] vLLM is using nccl==2.20.5
(RayWorkerWrapper pid=1918277) INFO 06-10 10:23:35 utils.py:618] Found nccl from library libnccl.so.2
(RayWorkerWra

In [4]:
messages = [{"role": "user",
             "content": f"""너는 자기소개서 첨삭 전문가야.
주어진 자기소개서를 분석해서 평가해야해.
참고할 사항은 다음과 같아
1. '저는', '제가', '저의'같은 1인칭 주어시첨 표현은 제한적으로 사용.
2. '귀사', '당사', '이 회사' 같은 표현보단 기업의 정식명칭 사용.
3. 문장 구조나 띄어쓰기에 대한 평가는 하지 않음.

출력은 다음의 형식을 따라
[평가]
이런 점이 좋아요:
n. (소제목):
n. (소제목):
...
다시 생각해 봐요:
n. (소제목):
n. (소제목):
...

다음이 자기소개서야 :
[겸손하게 배우는 자세]
평소 디지털 트랜스포메이션에 의해 비즈니스의 속도와 변화가 심해지면서 IT 개발 직무에서도 여러 가지의 역할을 할 수 있어야 한다고 생각했습니다. 따라서 저는 IT와 함께 소셜미디어 매니지먼트를 연계 전공하게 되었습니다. IT와 비즈니스 관련수업을 병행해온 저에게CJ오쇼핑은 일반 IT 회사와 다르게 유통회사의 IT 부서이었던 것이 매력적으로 다가왔습니다. 저는 연계 전공 동안 음식점을 마케팅한 적 있습니다. 음식점에서 단순히 음식을 파는 것이 아니라 고객의 입장에서 생각해보는 것이 중요하다는 것을 깨달았습니다. 또한, 미국을 여행하는 동안 떠오른 여행용 가방과 관련된 아이디어로 창업경진대회까지도 참여했습니다. 입상에는 실패했지만, 도전을 무서워하지 않는 용기를 가지게 되었습니다. 연계 전공을 하며 본 전공에 대해 이해도가 떨어질 수도 있지만, 항상 배우려는 태도로 임하는 겸손한 사람이 되겠습니다. 또한, 최근에는 유럽 16개국을 여행하며 다양한 인종의 사람들을 만나 동행을 하게 되면서 다른 사람의 마음을 잘 파악한다는 이야기를 많이 들었습니다. 평소 남들보다 먼저 행동하고 해결하려 했던 것들이 다른 사람들의 생각을 아는 데 도움이 되었던 것 같습니다. IT 개발 업무에서도 고객의 마음을 빠르게 파악하여 트랜드에 민감하게 반응하는 개발자가 되겠습니다.
제가 바라보는 CJ오쇼핑의 IT기술 강점은 트랜드에 굉장히 민감하게 반응한다는 것입니다. 최근 커머스와 미디어시장의 경제가 무너지고 미디어급변이 일어나면서. CJ오쇼핑은 E&M과 합병을 결정하였습니다. 그에 따라 최근 이슈인 인공지능 음성인식 서비스나 가상현실, 증강현실 등의 개발을 확대해가기에 새로운 경쟁력을 가진다고 생각합니다. 하지만 최근 T커머스에 주력을 다 하면서, 모바일 사업 부분에서 경쟁사들보다 부진하다고 생각합니다. 따라서 다양한 계열사를 가진 장점을 이용해 계열사 간 연계를 강화하고 모바일 최적화 상품과 동영상 콘텐츠를 확대하며 신규 서비스를 적극 추진한다면 고객 확보에 도움이 될 것이라고 생각합니다.
"""}]


In [5]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)


sampling_params = SamplingParams(temperature=0, max_tokens=4096)
outputs = llm.generate(text, sampling_params)
print(outputs[0].outputs[0].text)

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.39s/it, Generation Speed: 120.80 toks/s]

 [평가]
이런 점이 좋아요:
1. (다양한 경험):
   - IT와 소셜미디어 매니지먼트를 연계 전공한 경험을 통해 다양한 역할을 수행할 수 있는 능력을 강조한 점이 좋습니다.
   - 음식점 마케팅 경험과 창업경진대회 참여를 통해 도전 정신과 문제 해결 능력을 보여준 점이 긍정적입니다.
   - 유럽 여행 경험을 통해 다양한 인종의 사람들을 만나면서 다른 사람의 마음을 잘 파악하는 능력을 강조한 점이 좋습니다.

2. (기업 이해도):
   - CJ오쇼핑의 IT 부서 특성에 대한 이해를 바탕으로 지원 동기를 명확히 제시한 점이 좋습니다.
   - CJ오쇼핑의 최근 합병과 기술 강점을 언급하며 기업의 현재 상황에 대한 깊은 이해를 보여준 점이 긍정적입니다.

다시 생각해 봐요:
1. (1인칭 주어 사용):
   - "저는", "제가", "저의" 같은 1인칭 주어 사용이 빈번합니다. 이를 줄이고 더 객관적인 표현을 사용하면 좋겠습니다.

2. (기업 명칭 사용):
   - "CJ오쇼핑"이라는 정식 명칭을 사용한 점은 좋지만, "귀사", "당사" 같은 표현을 사용하지 않도록 주의해야 합니다.

3. (구체적인 기여 방안):
   - CJ오쇼핑의 모바일 사업 부진에 대한 분석과 개선 방안을 제시했지만, 구체적인 기여 방안을 더 명확히 제시하면 좋겠습니다. 예를 들어, 어떤 기술을 활용하고 어떤 방식으로 문제를 해결할 것인지 구체적으로 설명하면 더 설득력 있을 것입니다.

4. (겸손한 자세 강조):
   - 겸손한 자세를 강조하는 부분이 좋지만, 이를 통해 어떻게 구체적으로 기여할 수 있는지 더 명확히 설명하면 좋겠습니다. 예를 들어, 겸손한 자세로 어떤 문제를 해결하고 어떤 성과를 낼 수 있는지 구체적으로 언급하면 더 효과적일 것입니다.


In [7]:
from langchain_community.llms import VLLM

llm = VLLM(
    model="cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0",
    tensor_parallel_size=8,
    temperature=0,
    top_k=50,
    top_p=0.95,
    max_model_len=1024,
    vllm_kwargs={'gpu_memory_utilization': 0.2}
)

print(llm("model test"))

/home/ubuntu/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2024-06-10 10:45:57,036	INFO worker.py:1749 -- Started a local Ray instance.


INFO 06-10 10:45:58 llm_engine.py:161] Initializing an LLM engine (v0.4.3) with config: model='cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0', speculative_config=None, tokenizer='cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=8, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), seed=0, served_model_name=cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0)
INFO 06-10 10:46:21 utils.py:618] Found nccl from library libnccl.so.2
INFO 06-10 10:46:21 pynccl.py:65] vLLM is using nccl==2.20.5
(RayWorkerWrapper pid=1955612) INFO 06-10 10:46:21 utils.py:618] Found nccl from library libnccl.so.2
(RayWorkerWra

/home/ubuntu/.local/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(
Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.52s/it, Generation Speed: 120.70 toks/s]

는 모델의 성능을 테스트하는 과정입니다. 모델의 성능을 테스트하기 위해서는 모델의 예측값과 실제값을 비교하는 과정이 필요합니다. 모델의 예측값과 실제값을 비교하기 위해서는 모델의 성능을 측정하는 지표를 사용해야 합니다.
모델의 성능을 측정하는 지표로는 정확도, 정밀도, 재현율, F1-score 등이 있습니다. 정확도는 모델의 예측값이 실제값과 일치하는 비율을 나타내는 지표입니다. 정밀도는 모델이 긍정적으로 예측한 사례 중 실제로 긍정적인 사례의 비율을 나타내는 지표입니다. 재현율은 모델이 긍정적으로 예측한 사례 중 실제로 긍정적인 사례의 비율을 나타내는 지표입니다. F1-score는 정밀도와 재현율을 결합한 지표로, 모델의 전반적인 성능을 평가하는 데 사용됩니다.
모델의 성능을 테스트할 때는 모델의 예측값과 실제값을 비교하여 모델의 성능을 측정하는 지표를 계산합니다. 모델의 성능이 만족스럽지 않다면, 모델의 파라미터를 조정하거나 모델의 구조를 변경하여 성능을 개선할 수 있습니다.

모델의 성능을 테스트하는 과정은 모델의 성능을 향상시키는 중요한 단계입니다. 모델의 성능을 테스트하여 모델의 성능을 개선함으로써, 모델이 실제 상황에서 더 잘 작동할 수 있도록 할 수 있습니다.


(RayWorkerWrapper pid=1955612) ERROR 06-10 11:17:06 worker_base.py:148] Error executing method start_worker_execution_loop. This might cause deadlock in distributed execution.
(RayWorkerWrapper pid=1955612) ERROR 06-10 11:17:06 worker_base.py:148] Traceback (most recent call last):
(RayWorkerWrapper pid=1955612) ERROR 06-10 11:17:06 worker_base.py:148]   File "/home/ubuntu/.local/lib/python3.10/site-packages/vllm/worker/worker_base.py", line 140, in execute_method
(RayWorkerWrapper pid=1955612) ERROR 06-10 11:17:06 worker_base.py:148]     return executor(*args, **kwargs)
(RayWorkerWrapper pid=1955612) ERROR 06-10 11:17:06 worker_base.py:148]   File "/home/ubuntu/.local/lib/python3.10/site-packages/torch/utils/_contextlib.py", line 115, in decorate_context
(RayWorkerWrapper pid=1955612) ERROR 06-10 11:17:06 worker_base.py:148]     return func(*args, **kwargs)
(RayWorkerWrapper pid=1955612) ERROR 06-10 11:17:06 worker_base.py:148]   File "/home/ubuntu/.local/lib/python3.10/site-packages/

In [1]:
# 캐시 지우기
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel,TextStreamer, pipeline 
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA, LLMChain
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
import torch
import pandas as pd

def generate_talent_description(company_name, retriever, llm):
    # 질의 생성
    query = f""" 
    "회사의 모든 인재상을 입력하세요."
    "인재상 설명 이외의 다른 단어는 출력하지 마세요."
    "질문에 최선을 다해 답변하세요. "
    "검색 가능한 모든 도구를 자유롭게 사용하세요 "
    "필요한 경우에만 관련 정보 제공을 하세요"
    "답을 모르면 모른다고 말하세요. 답을 지어내려고 하지 마세요."
    "반드시 한국어로 답변하세요"

    [출력형식]
    {company_name}의 인재상
    1. 인재상1
    2. 인재상2
    3. 인재상3
    ...
    """
    
    # 답변 생성
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True
    )
    
    result = qa_chain({"query": query})
    talents = result["result"]

    print(talents)
    return talents

In [3]:
# 검색 관련 함수
def extract_company_info(company_name):
    # 크롬 옵션 설정
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    # WebDriver 설정
    driver = webdriver.Chrome(options=chrome_options)

    try:
        # 변수 설정
        base_url = "https://www.jobkorea.co.kr/starter/companyreport"

        # URL 구성 및 접속
        url = f"{base_url}?schTxt={company_name}"
        driver.get(url)

        # 원하는 요소가 로드될 때까지 기다리고 요소 찾기
        element = WebDriverWait(driver, 3).until(
            EC.presence_of_element_located((By.XPATH, f'//dt/a/strong[contains(text(), "기업심층분석 1. {company_name}, 채용분석 및 기업정보")]'))
        )
    
        href = element.find_element(By.XPATH, '..').get_attribute('href')
        print(f"Found detailed analysis link: {href}")

        # 링크로 이동
        driver.get(href)

        # 페이지 소스 가져오기
        page_source = driver.page_source

        # BeautifulSoup으로 페이지 소스 파싱
        soup = BeautifulSoup(page_source, 'html.parser')

        # 'viewWrap' 클래스를 가진 모든 요소 찾기
        view_wrap_elements = soup.find_all(class_="viewWrap")

        # 각 요소에서 텍스트 추출
        extracted_text = ""
        for element in view_wrap_elements:
            text = element.get_text(strip=True)
            extracted_text += text + "\n"

        return extracted_text

    except Exception as e:
        print(f"An error occurred: {e}")
        return None

    finally:
        driver.quit()

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": "cuda"})

/home/ubuntu/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:

# 변수 설정
company_name = "카카오"
report = """
안녕하세요. 저는 호남대학교 컴퓨터공학과 4학년에 재학 중인 이진수입니다. 저는 어려서부터 컴퓨터와 프로그래밍에 관심이 많았고, 대학에 입학한 후에는 소프트웨어 개발에 필요한 다양한 기술과 지식을 배우며 역량을 키워 왔습니다.
저는 학부 과정에서 프로그래밍 언어, 자료구조, 알고리즘, 데이터베이스, 웹 개발 등 컴퓨터공학의 기본 지식을 충실히 학습했습니다. 특히 Java와 Python을 활용한 프로젝트를 통해 실무에 적용 가능한 개발 기술을 연마했습니다. 3학년 때는 '학생 관리 시스템' 프로젝트에 참여하여 웹 애플리케이션 개발 전 과정을 경험하였고, 백엔드 개발을 담당하며 REST API 설계와 데이터베이스 관리 능력을 키웠습니다.
또한, 저는 꾸준한 자기개발을 통해 최신 기술 동향을 파악하고 역량을 강화하고 있습니다. 최근에는 클라우드 컴퓨팅과 머신러닝에 관심을 갖고 AWS와 TensorFlow를 학습하고 있습니다. 이러한 새로운 기술을 프로젝트에 적극적으로 활용하여 혁신적인 솔루션을 개발하는 것이 저의 목표입니다.
"""

# CSV 파일 경로
csv_file_path = "./company_talents.csv"

# CSV 파일 읽기
df = pd.read_csv(csv_file_path)

# company_name 열에 company_name이 있는지 확인
if company_name in df["company_name"].values:
    # company_name이 존재하는 경우, talents 열의 값을 가져옴
    talents = df[df["company_name"] == company_name]["talents"].tolist()
    # 결과 출력
    print("Talents:", talents)
else:
    # company_name이 존재하지 않는 경우, extract_company_info 함수 실행
    company_info = extract_company_info(company_name)
    if company_info:
        # 문서 임베딩
        vector_store = FAISS.from_texts(texts=[company_info], embedding=embeddings)

        # Retriever 생성
        retriever = vector_store.as_retriever(search_kwargs={"k": 1})

        # 인재상 생성
        talents = generate_talent_description(company_name, retriever, llm)
        print(talents)
        str(talents)

        # 결과를 데이터프레임에 추가
        new_data = {"company_name": company_name, "talents": talents}
        df = pd.concat([df, pd.DataFrame([new_data])], ignore_index=True)
        
        # 결과를 CSV 파일에 저장
        df.to_csv(csv_file_path, index=False)
        print("인재상이 생성되어 CSV 파일에 추가되었습니다.")
    else:
        print("기업 정보 추출에 실패했습니다. 직접 인재상을 작성해주세요")
        # 인재상 입력
        talents = input("인재상 : ")
        
        # 결과를 데이터프레임에 추가
        new_data = {"company_name": company_name, "talents": talents}
        df = pd.concat([df, pd.DataFrame([new_data])], ignore_index=True)

        # 결과를 CSV 파일에 저장
        df.to_csv(csv_file_path, index=False)
        print("인재상이 생성되어 CSV 파일에 추가되었습니다.")

Talents: ['\n 카카오의 인재상은 다음과 같습니다:\n\n1. Willing to Venture: 가보지 않은 길을 두려워하지 않습니다.\n2. Back to Basic: 무엇이든 본질만 남기고 처음부터 다시 생각해 봅니다.\n3. Trust to Trust: 나보다 동료의 생각이 더 옳을 수 있다는 믿음을 가집니다.\n4. Act for Yourself: 스스로 몰입하고 주도적으로 일합니다.\n5. Tech for Good: 세상을 선하게 바꾸려고 노력합니다.']


In [9]:
prompt_template = f"""
당신은 인재상을 기준으로 사용자의 자기소개서를 수정하는 역할을 맡고 있습니다. 아래의 첨삭과정을 지키면서 작업을 수행하세요.

[입력 정보]
기업명: {company_name}
인재상: {talents}
자기소개서: {report}

[첨삭 과정]
1. {company_name}의 인재상을 확인하세요.
2. 제공된 자기소개서의 내용과 흐름을 분석하세요.
3. 자기소개서에서 지원자의 인성 및 역량 중 인재상과 부합하는 부분을 식별하세요.
4. 식별된 부분을 강조하고, 해당 문단에서 인재상과 관련된 경험이나 역량을 구체적으로 설명하세요.
5. 모든 문단에 걸쳐 인재상이 고르게 반영되도록 하세요. 
6. 한 문단에 인재상 관련 내용이 집중되지 않도록 주의하세요.
7. 기업명 '{company_name}'을 포함한 모든 고유명사를 'OOO'로 대체하세요.
8. 수정된 문단들을 하나의 자기소개서로 통합하세요.

[출력 형식]
수정된 자기소개서: (수정된 자기소개서 내용)
"""

# 언어 모델 실행
result = llm(prompt=prompt_template)
print(result)

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, Generation Speed: 0.00 toks/s]

ERROR 06-10 11:26:29 worker_base.py:148] Error executing method execute_model. This might cause deadlock in distributed execution.
ERROR 06-10 11:26:29 worker_base.py:148] Traceback (most recent call last):
ERROR 06-10 11:26:29 worker_base.py:148]   File "/home/ubuntu/.local/lib/python3.10/site-packages/vllm/worker/worker_base.py", line 140, in execute_method
ERROR 06-10 11:26:29 worker_base.py:148]     return executor(*args, **kwargs)
ERROR 06-10 11:26:29 worker_base.py:148]   File "/home/ubuntu/.local/lib/python3.10/site-packages/torch/utils/_contextlib.py", line 115, in decorate_context
ERROR 06-10 11:26:29 worker_base.py:148]     return func(*args, **kwargs)
ERROR 06-10 11:26:29 worker_base.py:148]   File "/home/ubuntu/.local/lib/python3.10/site-packages/vllm/worker/worker.py", line 264, in execute_model
ERROR 06-10 11:26:29 worker_base.py:148]     broadcast_tensor_dict(data, src=0)
ERROR 06-10 11:26:29 worker_base.py:148]   File "/home/ubuntu/.local/lib/python3.10/site-packages/vl

RuntimeError: [../third_party/gloo/gloo/transport/tcp/pair.cc:534] Connection closed by peer [127.0.1.1]:46988